In [1]:
# Notebook: 02_gini_sampling_distribution
# Monte Carlo sampling behaviour of Metric II (the stratified directional Gini).
# Under an even-spread DGP fixed to the disclosed default rates and direction mix,
# this notebook estimates the bias, standard error, and interval coverage of the
# Gini-dir estimator and its upgrade/downgrade arms as the total sample size grows,
# establishing that the one-number summary is consistent and how large a sample a
# stable estimate needs.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG = np.random.default_rng(20260207)
BASE = os.path.join("..")
FIG = os.path.join(BASE, "results", "figures")
TAB = os.path.join(BASE, "results", "tables")

# --- grayscale house style; no figure captions ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.30",
                     "axes.linewidth": 0.8, "grid.color": "0.88",
                     "grid.linewidth": 0.6, "savefig.bbox": "tight",
                     "figure.dpi": 150})


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, name + "." + ext), dpi=600)
    plt.close(fig)


# --- DGP: disclosed rates (Table 4) and direction mix ---
rates = pd.read_csv(os.path.join(BASE, "data", "disclosed_rates.csv"))
rates["p"] = rates["default_rate_pct"] / 100.0
P = {(r.band, r.direction): (np.nan if pd.isna(r.p) else float(r.p))
     for r in rates.itertuples()}
mix = pd.read_csv(os.path.join(BASE, "data", "direction_mix.csv"))
PI = {r.direction: r.proportion_pct / 100.0 for r in mix.itertuples()}
BANDS = ["BB+", "BB", "BB-", "B+", "B", "B-"]
G = len(BANDS)
ORDER = {"U": 1, "H": 2, "D": 3}  # ascending risk rank


# --- stratified directional Gini from defaulter/non-defaulter counts per cell ---
def gini_dir(a, b, arms=("U", "H", "D")):
    # a[(g,d)] defaulters, b[(g,d)] non-defaulters; concordant = defaulter in the
    # higher-risk direction paired with a non-defaulter in the lower-risk direction.
    C = Dd = T = 0.0
    seq = [d for d in ("U", "H", "D") if d in arms]
    for g in BANDS:
        for i in range(len(seq)):
            for j in range(len(seq)):
                di, dj = seq[i], seq[j]
                if ORDER[di] < ORDER[dj]:
                    C += a[(g, dj)] * b[(g, di)]
                    Dd += a[(g, di)] * b[(g, dj)]
        for d in seq:
            T += a[(g, d)] * b[(g, d)]
    denom = C + Dd + T
    return np.nan if denom == 0 else (C - Dd) / denom


# --- true values: plug expected counts (no sampling) under even spread ---
def expected_counts(Ntot):
    a, b = {}, {}
    for g in BANDS:
        for d in ("U", "H", "D"):
            p = P[(g, d)]
            n = Ntot * PI[d] / G
            if np.isnan(p):
                a[(g, d)] = 0.0
                b[(g, d)] = 0.0
            else:
                a[(g, d)] = n * p
                b[(g, d)] = n * (1 - p)
    return a, b


a0, b0 = expected_counts(1e9)
TRUE = {"Gini_dir": gini_dir(a0, b0),
        "Gini_U": gini_dir(a0, b0, ("U", "H")),
        "Gini_D": gini_dir(a0, b0, ("H", "D"))}

# --- Monte Carlo over total sample size ---
N_GRID = [2000, 5000, 10000, 20000, 50000, 100000]
R = 4000
records = {"Gini_dir": {}, "Gini_U": {}, "Gini_D": {}}
draws_store = {}
for Ntot in N_GRID:
    ncell = {(g, d): max(1, int(round(Ntot * PI[d] / G))) for g in BANDS
             for d in ("U", "H", "D")}
    est = {"Gini_dir": np.empty(R), "Gini_U": np.empty(R), "Gini_D": np.empty(R)}
    for r in range(R):
        a, b = {}, {}
        for g in BANDS:
            for d in ("U", "H", "D"):
                p = P[(g, d)]
                if np.isnan(p):
                    a[(g, d)] = 0.0
                    b[(g, d)] = 0.0
                else:
                    k = RNG.binomial(ncell[(g, d)], p)
                    a[(g, d)] = k
                    b[(g, d)] = ncell[(g, d)] - k
        est["Gini_dir"][r] = gini_dir(a, b)
        est["Gini_U"][r] = gini_dir(a, b, ("U", "H"))
        est["Gini_D"][r] = gini_dir(a, b, ("H", "D"))
    for key in records:
        records[key][Ntot] = est[key]
    draws_store[Ntot] = est["Gini_dir"]

# --- summary table: bias, SE, 95% interval, coverage of the true value ---
sum_rows = []
for key in ("Gini_dir", "Gini_U", "Gini_D"):
    for Ntot in N_GRID:
        v = records[key][Ntot]
        lo, hi = np.percentile(v, [2.5, 97.5])
        sum_rows.append({"statistic": key, "N_total": Ntot,
                         "true": round(TRUE[key], 4),
                         "mean_est": round(float(np.mean(v)), 4),
                         "bias": round(float(np.mean(v) - TRUE[key]), 4),
                         "se": round(float(np.std(v, ddof=1)), 4),
                         "ci_lo": round(float(lo), 4),
                         "ci_hi": round(float(hi), 4),
                         "covers_true": bool(lo <= TRUE[key] <= hi)})
summary_df = pd.DataFrame(sum_rows)
summary_df.to_csv(os.path.join(TAB, "t02_gini_sampling_summary.csv"), index=False)

# --- figure 3: sampling distribution of Gini-dir at a small vs large sample ---
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for Ntot, shade in [(2000, "0.65"), (20000, "0.10")]:
    sns.kdeplot(draws_store[Ntot], ax=ax, color=shade, linewidth=1.6,
                fill=True, alpha=0.18, label=f"N = {Ntot:,}")
ax.axvline(TRUE["Gini_dir"], color="0.30", linewidth=1.0, linestyle=(0, (4, 2)))
ax.set_xlabel(r"Estimated stratified directional Gini")
ax.set_ylabel("Density")
ax.legend(frameon=False, fontsize=9)
save_fig(fig, "fig03_gini_sampling_distribution")

# --- figure 4: standard error vs total sample size (log-log, grayscale) ---
se_by_N = [np.std(records["Gini_dir"][Ntot], ddof=1) for Ntot in N_GRID]
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.plot(N_GRID, se_by_N, color="0.10", marker="o", markersize=5,
        markerfacecolor="white", markeredgecolor="0.10", linewidth=1.4)
ref = se_by_N[0] * np.sqrt(N_GRID[0] / np.array(N_GRID))
ax.plot(N_GRID, ref, color="0.55", linewidth=1.0, linestyle=(0, (1, 1)),
        label=r"$\propto N^{-1/2}$ reference")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total firms (log scale)")
ax.set_ylabel("Standard error of Gini-dir (log scale)")
ax.legend(frameon=False, fontsize=9)
save_fig(fig, "fig04_gini_standard_error")

print("True values under the even-spread DGP:")
for k, v in TRUE.items():
    print(f"  {k}: {v:+.4f}")
print("\nSampling summary (Gini-dir):")
print(summary_df[summary_df['statistic'] == 'Gini_dir'].to_string(index=False))


True values under the even-spread DGP:
  Gini_dir: +0.0715
  Gini_U: +0.0088
  Gini_D: +0.0699

Sampling summary (Gini-dir):
statistic  N_total   true  mean_est    bias     se   ci_lo  ci_hi  covers_true
 Gini_dir     2000 0.0715    0.0709 -0.0005 0.0831 -0.0939 0.2353         True
 Gini_dir     5000 0.0715    0.0713 -0.0002 0.0517 -0.0302 0.1735         True
 Gini_dir    10000 0.0715    0.0714 -0.0001 0.0365 -0.0022 0.1438         True
 Gini_dir    20000 0.0715    0.0711 -0.0004 0.0258  0.0213 0.1207         True
 Gini_dir    50000 0.0715    0.0716  0.0001 0.0158  0.0404 0.1026         True
 Gini_dir   100000 0.0715    0.0713 -0.0001 0.0114  0.0494 0.0935         True
